# Colab L4: memoria en pesos en `LiquidAI/LFM3-1.7B`, profundidad completa + mas pasos

La corrida anterior (capas 0-2, 40 pasos, bf16) dio **87%** con el full-name gate escribiendo en L0, pero la escritura no convergio (NLL final `0.75`, meta `0.05`).

Esta corrida contesta dos preguntas a la vez:
1. **Profundidad:** barrido `0, 6, 12, 18, 23` (de 24). WISE/MEMOIR dicen que lo mejor esta a media o tardia profundidad en modelos de 7B; en LFM3 solo probamos L0-L10.
2. **Convergencia:** `--steps 120` (antes 40) y el modelo en `fp32` (antes bf16).

Resume en Drive (`MyDrive/wsi_runs/moi_memory_lfm3_depth/progress.pt`), resultado final en `MyDrive/wsi_runs/FINALES/moi_memory_lfm3_100_depth_steps_l4.json`, y se apaga solo al terminar. ~30-45 min en L4.


In [ ]:
MODEL = "LiquidAI/LFM3-1.7B"  # Alternativa: "Qwen/Qwen3.5-4B" (ajustar WRITE_BATCH=8 si hace falta)
DTYPE = "fp32"
RUN_NAME = "moi_memory_lfm3_100_depth_steps_l4"
REPO = "/content/wsi"
BASE = "/content/base"
DRIVE_DIR = "/content/drive/MyDrive/wsi_runs/moi_memory_lfm3_depth"
FINALES_OUT = f"/content/drive/MyDrive/wsi_runs/FINALES/{RUN_NAME}.json"
DRIVE_OUT = f"{DRIVE_DIR}/results_l4.json"
PROGRESS_PT = f"{DRIVE_DIR}/progress.pt"
SAVE_BANK = f"{DRIVE_DIR}/lfm3_100_depth_steps.moimem"
PINNED_COMMIT = "f828c2b1820b36c4c8fa531c8b851542f90e0ba9"

PEOPLE = "10,100"          # Subir a "10,100,500" o "10,100,1000" para curva completa
EVAL_PEOPLE = 50
LAYERS = "0,6,12,18,23"   # profundidad completa (24 capas)
STEPS = 120               # antes 40
WRITE_BATCH = 8            # fp32 usa el doble de memoria (el script lo baja solo si hay OOM)
ARMS = "base,ctx,rome,rome_gated,rome_fullname,memit"

In [ ]:
import subprocess
import sys

subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], check=False)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "torch", "transformers", "accelerate", "safetensors", "datasets", "numpy", "huggingface_hub"],
    check=True,
)

import datasets, safetensors, torch, transformers

print("torch", torch.__version__, "| transformers", transformers.__version__, "| datasets", datasets.__version__, "| safetensors", safetensors.__version__)

In [ ]:
import os
import subprocess
from google.colab import drive
from huggingface_hub import snapshot_download

drive.mount("/content/drive")
os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs(os.path.dirname(FINALES_OUT), exist_ok=True)

if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/krexito/weight-space-index.git", REPO], check=True)
subprocess.run(["git", "-C", REPO, "fetch", "--depth", "1", "origin", "main"], check=True)
subprocess.run(["git", "-C", REPO, "reset", "--hard", "origin/main"], check=True)
COMMIT = subprocess.check_output(["git", "-C", REPO, "rev-parse", "HEAD"], text=True).strip()
print("Repo commit:", COMMIT, "(esperado >= ", PINNED_COMMIT[:7], ")")

if not os.path.exists(BASE) or not os.listdir(BASE):
    snapshot_download(MODEL, local_dir=BASE)
print("Base model ready:", BASE)

In [ ]:
import datetime
import json
import os
import subprocess
import sys

SCRIPT = f"{REPO}/scripts/moi_memory.py"
if not os.path.exists(SCRIPT):
    raise FileNotFoundError(f"Falta {SCRIPT} en {REPO} (commit {COMMIT}).")

cmd = [
    sys.executable, "-u", SCRIPT,
    "--model", BASE,
    "--dtype", DTYPE,
    "--people", PEOPLE,
    "--eval-people", str(EVAL_PEOPLE),
    "--layers", LAYERS,
    "--steps", str(STEPS),
    "--write-batch", str(WRITE_BATCH),
    "--arms", ARMS,
    "--progress", PROGRESS_PT,
    "--save-bank", SAVE_BANK,
    "--tf32",
    "--out", DRIVE_OUT,
]

env = dict(os.environ)
env["PYTHONPATH"] = REPO + (os.pathsep + env["PYTHONPATH"] if env.get("PYTHONPATH") else "")

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
assert proc.stdout is not None
for line in proc.stdout:
    print(line, end="", flush=True)
rc = proc.wait()
if rc != 0:
    raise subprocess.CalledProcessError(rc, cmd)

with open(DRIVE_OUT, encoding="utf-8") as f:
    res = json.load(f)

res["_final"] = {
    "complete": True,
    "model": MODEL,
    "dtype": DTYPE,
    "commit": COMMIT,
    "notebook": "moi_memory_lfm3_depth_steps_colab.ipynb",
    "steps": STEPS,
    "layers_swept": LAYERS,
    "gpu": subprocess.check_output(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True).strip(),
    "saved_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
}

tmp_final = FINALES_OUT + ".tmp"
with open(tmp_final, "w", encoding="utf-8") as f:
    json.dump(res, f, indent=2, ensure_ascii=False)
os.replace(tmp_final, FINALES_OUT)
print("\nCopia final atomica guardada en:", FINALES_OUT)

In [ ]:
import json

with open(FINALES_OUT, encoding="utf-8") as f:
    res = json.load(f)

print(f"Layer elegida: {res.get('layer')} | Duracion: {res.get('minutes')} min | Base WikiText PPL: {res.get('base_ppl', {}).get('ppl')} | Capitals base: {res.get('capitals_known_by_base')}/30")

keys = ["write", "dev", "para", "es", "chat", "hop", "nb_last"]
for scale_k, sc in res.get("scales", {}).items():
    print(f"\n=== Escala N={scale_k} memorias (eval_people={sc.get('n_eval_people')}) ===")
    hdr = f"{'Arm':14s} | " + " | ".join(f"{k:11s}" for k in keys) + " | WikiPPL | FireRate | Caps"
    print(hdr)
    print("-" * len(hdr))
    for arm_name, arm_v in sc.get("arms", {}).items():
        pr = arm_v.get("probes", {})
        cells = []
        for k in keys:
            if k in pr:
                ch = pr[k].get("choice", 0.0) * 100
                ex = pr[k].get("exact", 0.0) * 100
                cells.append(f"{ch:4.0f}%/{ex:4.0f}%")
            else:
                cells.append("    ---    ")
        ppl = arm_v.get("ppl", {}).get("ppl")
        fire = arm_v.get("ppl", {}).get("fire_token_rate")
        caps = arm_v.get("capitals_retained")
        ppl_s = f"{ppl:7.2f}" if isinstance(ppl, (int, float)) else "    ---"
        fire_s = f"{fire*100:6.2f}% " if isinstance(fire, (int, float)) else "     ---"
        caps_s = f"{caps*100:4.0f}%" if isinstance(caps, (int, float)) else "  ---"
        print(f"{arm_name:14s} | " + " | ".join(cells) + f" | {ppl_s} | {fire_s} | {caps_s}")

try:
    from google.colab import files
    files.download(FINALES_OUT)
except Exception as exc:
    print("Descarga manual desde Drive:", FINALES_OUT, f"({exc})")

In [ ]:
AUTO_APAGAR = True
import json, os, time
ok = os.path.exists(FINALES_OUT) and json.load(open(FINALES_OUT, encoding="utf-8")).get("_final", {}).get("complete")
if AUTO_APAGAR and ok:
    time.sleep(60)
    from google.colab import runtime
    runtime.unassign()
else:
    print("No apago: el resultado todavia no esta completo en FINALES.")